In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# imports

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: "%.4f" % x)
print("Imports OK ✓")

Imports OK ✓


# load data

In [4]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")

train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Fraud rate: {train['isFraud'].mean():.4f} ({train['isFraud'].sum()} fraud out of {len(train)} total)")

Train: (590540, 434)
Test:  (506691, 433)
Fraud rate: 0.0350 (20663 fraud out of 590540 total)


# target distribution

In [6]:
print("=== TARGET DISTRIBUTION ===")
fraud_counts = train["isFraud"].value_counts()
print(fraud_counts)
print(f"\nFraud: {fraud_counts[1]} ({fraud_counts[1]/len(train)*100:.2f}%)")
print(f"Non-fraud: {fraud_counts[0]} ({fraud_counts[0]/len(train)*100:.2f}%)")
print(f"\nClass imbalance ratio: {fraud_counts[0]/fraud_counts[1]:.1f}:1")


=== TARGET DISTRIBUTION ===
isFraud
0    569877
1     20663
Name: count, dtype: int64

Fraud: 20663 (3.50%)
Non-fraud: 569877 (96.50%)

Class imbalance ratio: 27.6:1


# MISSING VALUE ANALYSIS

In [7]:
print("=== MISSING VALUES ANALYSIS ===")
missing = train.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]

print(f"Columns with missing values: {len(missing)} out of {train.shape[1]}")
print(f"\nMissing rate distribution:")
print(f"  >90% missing: {(missing > 0.9).sum()} cols")
print(f"  >50% missing: {(missing > 0.5).sum()} cols")
print(f"  >30% missing: {(missing > 0.3).sum()} cols")
print(f"  >10% missing: {(missing > 0.1).sum()} cols")

print(f"\nTop 20 most missing columns:")
print(missing.head(20))


=== MISSING VALUES ANALYSIS ===
Columns with missing values: 414 out of 434

Missing rate distribution:
  >90% missing: 12 cols
  >50% missing: 214 cols
  >30% missing: 232 cols
  >10% missing: 322 cols

Top 20 most missing columns:
id_24   0.9920
id_25   0.9913
id_07   0.9913
id_08   0.9913
id_21   0.9913
id_26   0.9913
id_27   0.9912
id_23   0.9912
id_22   0.9912
dist2   0.9363
D7      0.9341
id_18   0.9236
D13     0.8951
D14     0.8947
D12     0.8904
id_04   0.8877
id_03   0.8877
D6      0.8761
id_33   0.8759
id_09   0.8731
dtype: float64


In [9]:
print("=== TRANSACTION AMOUNT ANALYSIS ===")
print(f"TransactionAmt stats:")
print(train["TransactionAmt"].describe())

print(f"\nPercentiles:")
for p in [50, 75, 90, 95, 99, 99.9]:
    print(f"  {p}th percentile: {train['TransactionAmt'].quantile(p/100):.2f}")

print(f"\nFraud vs non-fraud amounts:")
print(f"  Fraud mean:     {train[train['isFraud']==1]['TransactionAmt'].mean():.2f}")
print(f"  Non-fraud mean: {train[train['isFraud']==0]['TransactionAmt'].mean():.2f}")
print(f"  Fraud median:   {train[train['isFraud']==1]['TransactionAmt'].median():.2f}")
print(f"  Non-fraud median: {train[train['isFraud']==0]['TransactionAmt'].median():.2f}")

# round amount analysis
round_fraud = train[train["TransactionAmt"] % 1 == 0]["isFraud"].mean()
nonround_fraud = train[train["TransactionAmt"] % 1 != 0]["isFraud"].mean()
print(f"\nRound amount fraud rate:     {round_fraud:.4f}")
print(f"Non-round amount fraud rate: {nonround_fraud:.4f}")

print(f"Round amounts have {'higher' if round_fraud > nonround_fraud else 'lower'} fraud rate.")

=== TRANSACTION AMOUNT ANALYSIS ===
TransactionAmt stats:
count   590540.0000
mean       135.0272
std        239.1625
min          0.2510
25%         43.3210
50%         68.7690
75%        125.0000
max      31937.3910
Name: TransactionAmt, dtype: float64

Percentiles:
  50th percentile: 68.77
  75th percentile: 125.00
  90th percentile: 275.29
  95th percentile: 445.00
  99th percentile: 1104.00
  99.9th percentile: 2769.81

Fraud vs non-fraud amounts:
  Fraud mean:     149.24
  Non-fraud mean: 134.51
  Fraud median:   75.00
  Non-fraud median: 68.50

Round amount fraud rate:     0.0357
Non-round amount fraud rate: 0.0343
Round amounts have higher fraud rate.


In [10]:
print("=== TIME ANALYSIS ===")
print(f"TransactionDT range: {train['TransactionDT'].min()} to {train['TransactionDT'].max()}")
print(f"Span in days: {(train['TransactionDT'].max() - train['TransactionDT'].min()) / 86400:.0f} days")
print(f"Approx 6 months of data")

# fraud rate over time
train["TransactionDay"] = (train["TransactionDT"] / 86400).astype(int)
daily_fraud = train.groupby("TransactionDay")["isFraud"].mean()

print(f"\nDaily fraud rate stats:")
print(f"  Mean:  {daily_fraud.mean():.4f}")
print(f"  Min:   {daily_fraud.min():.4f}")
print(f"  Max:   {daily_fraud.max():.4f}")
print(f"  Std:   {daily_fraud.std():.4f}")

# hour of day analysis
train["TransactionHour"] = ((train["TransactionDT"] / 3600) % 24).astype(int)
hourly_fraud = train.groupby("TransactionHour")["isFraud"].mean()
peak_hour = hourly_fraud.idxmax()
print(f"\nHourly fraud rate:")
print(f"  Peak fraud hour: {peak_hour}:00 ({hourly_fraud[peak_hour]:.4f} fraud rate)")
print(f"  Lowest fraud hour: {hourly_fraud.idxmin()}:00 ({hourly_fraud.min():.4f} fraud rate)")

# train vs test time overlap
print(f"\nTrain TransactionDT: {train['TransactionDT'].min()} to {train['TransactionDT'].max()}")
print(f"Test  TransactionDT: {test_tr['TransactionDT'].min()} to {test_tr['TransactionDT'].max()}")
overlap = train['TransactionDT'].max() > test_tr['TransactionDT'].min()
print(f"Time overlap between train and test: {overlap}")



=== TIME ANALYSIS ===
TransactionDT range: 86400 to 15811131
Span in days: 182 days
Approx 6 months of data

Daily fraud rate stats:
  Mean:  0.0360
  Min:   0.0110
  Max:   0.0699
  Std:   0.0100

Hourly fraud rate:
  Peak fraud hour: 7:00 (0.1061 fraud rate)
  Lowest fraud hour: 13:00 (0.0229 fraud rate)

Train TransactionDT: 86400 to 15811131
Test  TransactionDT: 18403224 to 34214345
Time overlap between train and test: False


In [11]:
print("=== CLIENT (UID) ANALYSIS ===")

# create D1n — day card began
train["D1n"] = train["TransactionDay"] - train["D1"]
train["uid"] = (
    train["card1"].astype(str) + "_" +
    train["addr1"].astype(str) + "_" +
    train["D1n"].fillna(-1).round(0).astype(int).astype(str)
)

uid_fraud = train.groupby("uid")["isFraud"].agg(["mean", "count", "sum"])
uid_fraud.columns = ["fraud_rate", "tx_count", "fraud_count"]

print(f"\nTotal unique UIDs (clients): {len(uid_fraud)}")
print(f"UIDs with 2+ transactions: {(uid_fraud['tx_count'] >= 2).sum()}")

multi_tx = uid_fraud[uid_fraud["tx_count"] >= 2]
always_fraud    = (multi_tx["fraud_rate"] == 1.0).sum()
never_fraud     = (multi_tx["fraud_rate"] == 0.0).sum()
mixed           = ((multi_tx["fraud_rate"] > 0) & (multi_tx["fraud_rate"] < 1)).sum()

print(f"\nAmong clients with 2+ transactions:")
print(f"  Always fraud (100%):  {always_fraud} ({always_fraud/len(multi_tx)*100:.1f}%)")
print(f"  Never fraud (0%):     {never_fraud} ({never_fraud/len(multi_tx)*100:.1f}%)")
print(f"  Mixed (some fraud):   {mixed} ({mixed/len(multi_tx)*100:.1f}%)")

print(f"\nConclusion: {always_fraud/len(multi_tx)*100:.1f}% of multi-transaction clients")


=== CLIENT (UID) ANALYSIS ===

Total unique UIDs (clients): 217779
UIDs with 2+ transactions: 92599

Among clients with 2+ transactions:
  Always fraud (100%):  1986 (2.1%)
  Never fraud (0%):     87498 (94.5%)
  Mixed (some fraud):   3115 (3.4%)

Conclusion: 2.1% of multi-transaction clients


In [13]:
print("=== C COLUMNS ANALYSIS ===")
c_cols = [f"C{i}" for i in range(1, 15) if f"C{i}" in train.columns]
print(f"C columns found: {c_cols}")

corr = train[c_cols].corr()
print(f"\nCorrelation matrix (showing high correlations):")
for i in range(len(c_cols)):
    for j in range(i+1, len(c_cols)):
        c = corr.iloc[i, j]
        if abs(c) > 0.7:
            print(f"  {c_cols[i]} vs {c_cols[j]}: {c:.4f}")

print(f"\nFraud rate by C1 quartile:")
train["C1_quartile"] = pd.qcut(train["C1"], q=4, labels=["Q1","Q2","Q3","Q4"], duplicates="drop")
print(train.groupby("C1_quartile")["isFraud"].mean())



=== C COLUMNS ANALYSIS ===
C columns found: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']

Correlation matrix (showing high correlations):
  C1 vs C2: 0.9951
  C1 vs C4: 0.9678
  C1 vs C6: 0.9822
  C1 vs C7: 0.9263
  C1 vs C8: 0.9677
  C1 vs C10: 0.9582
  C1 vs C11: 0.9965
  C1 vs C12: 0.9279
  C1 vs C13: 0.7746
  C1 vs C14: 0.9518
  C2 vs C4: 0.9721
  C2 vs C6: 0.9748
  C2 vs C7: 0.9389
  C2 vs C8: 0.9759
  C2 vs C10: 0.9706
  C2 vs C11: 0.9939
  C2 vs C12: 0.9403
  C2 vs C13: 0.7512
  C2 vs C14: 0.9361
  C4 vs C6: 0.9623
  C4 vs C7: 0.8951
  C4 vs C8: 0.9600
  C4 vs C10: 0.9525
  C4 vs C11: 0.9745
  C4 vs C12: 0.8946
  C4 vs C14: 0.9077
  C5 vs C9: 0.9258
  C5 vs C13: 0.7175
  C6 vs C7: 0.8586
  C6 vs C8: 0.9220
  C6 vs C10: 0.9144
  C6 vs C11: 0.9911
  C6 vs C12: 0.8582
  C6 vs C13: 0.8085
  C6 vs C14: 0.9842
  C7 vs C8: 0.9830
  C7 vs C10: 0.9851
  C7 vs C11: 0.9152
  C7 vs C12: 0.9995
  C7 vs C14: 0.7947
  C8 vs C10: 0.9970
  C8 vs C11: 

ValueError: Bin labels must be one fewer than the number of bin edges

In [15]:
print("=== D COLUMNS ANALYSIS ===")
d_cols = [f"D{i}" for i in range(1, 16) if f"D{i}" in train.columns]
print(f"D columns: {d_cols}")

print(f"\nD1 stats (days since card began):")
print(train["D1"].describe())

print(f"\nD1n = TransactionDay - D1 (day card began, should be constant per client):")
print(train.groupby("uid")["D1n"].std().describe())
print("(Low std confirms D1n is nearly constant per client — good UID component)")

print(f"\nFraud rate by D1 quartile:")
train["D1_quartile"] = pd.qcut(train["D1"].fillna(-1), q=4, labels=["Q1","Q2","Q3","Q4"], duplicates="drop")
print(train.groupby("D1_quartile")["isFraud"].mean())



=== D COLUMNS ANALYSIS ===
D columns: ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15']

D1 stats (days since card began):
count   589271.0000
mean        94.3476
std        157.6604
min          0.0000
25%          0.0000
50%          3.0000
75%        122.0000
max        640.0000
Name: D1, dtype: float64

D1n = TransactionDay - D1 (day card began, should be constant per client):
count   92431.0000
mean        0.0000
std         0.0000
min         0.0000
25%         0.0000
50%         0.0000
75%         0.0000
max         0.0000
Name: D1n, dtype: float64
(Low std confirms D1n is nearly constant per client — good UID component)

Fraud rate by D1 quartile:
D1_quartile
Q1   0.0421
Q2   0.1137
Q3   0.0324
Q4   0.0147
Name: isFraud, dtype: float64


In [17]:
print("=== V COLUMNS ANALYSIS ===")
v_cols = [c for c in train.columns if c.startswith("V") and c[1:].isdigit()]
print(f"Total V columns: {len(v_cols)}")

# missing rate
v_missing = train[v_cols].isnull().mean()
print(f"\nV column missing rates:")
print(f"  0% missing:   {(v_missing == 0).sum()} cols")
print(f"  <50% missing: {(v_missing < 0.5).sum()} cols")
print(f"  >50% missing: {(v_missing > 0.5).sum()} cols")
print(f"  >90% missing: {(v_missing > 0.9).sum()} cols")

# top predictive V columns
print(f"\nTop 10 most predictive V columns (fraud rate correlation):")
v_corr = {}
for col in v_cols[:50]:  # check first 50 to save time
    if train[col].isnull().mean() < 0.5:
        try:
            c = abs(train[col].corr(train["isFraud"]))
            v_corr[col] = c
        except:
            pass

top_v = sorted(v_corr.items(), key=lambda x: x[1], reverse=True)[:10]
for col, corr_val in top_v:
    print(f"  {col}: {corr_val:.4f}")



=== V COLUMNS ANALYSIS ===
Total V columns: 339

V column missing rates:
  0% missing:   0 cols
  <50% missing: 180 cols
  >50% missing: 159 cols
  >90% missing: 0 cols

Top 10 most predictive V columns (fraud rate correlation):
  V45: 0.2818
  V44: 0.2604
  V40: 0.2124
  V39: 0.2031
  V38: 0.1990
  V43: 0.1983
  V42: 0.1894
  V33: 0.1835
  V17: 0.1827
  V18: 0.1825


In [19]:
print("=== CARD FEATURES ANALYSIS ===")

print("card4 (card network) fraud rates:")
print(train.groupby("card4")["isFraud"].agg(["mean", "count"]).sort_values("mean", ascending=False))

print("\ncard6 (card type) fraud rates:")
print(train.groupby("card6")["isFraud"].agg(["mean", "count"]).sort_values("mean", ascending=False))

print("\ncard1 (card issuer) — top 10 by fraud rate (min 100 transactions):")
card1_stats = train.groupby("card1")["isFraud"].agg(["mean", "count"])
card1_stats = card1_stats[card1_stats["count"] >= 100].sort_values("mean", ascending=False)
print(card1_stats.head(10))

print("\ncard1 transaction count distribution:")
print(train["card1"].value_counts().describe())



=== CARD FEATURES ANALYSIS ===
card4 (card network) fraud rates:
                   mean   count
card4                          
discover         0.0773    6651
visa             0.0348  384767
mastercard       0.0343  189217
american express 0.0287    8328

card6 (card type) fraud rates:
                  mean   count
card6                         
credit          0.0668  148986
debit           0.0243  439938
charge card     0.0000      15
debit or credit 0.0000      30

card1 (card issuer) — top 10 by fraud rate (min 100 transactions):
        mean  count
card1              
3643  0.5207    121
17999 0.4851    101
2939  0.4457    175
2801  0.3796    411
3702  0.3459    133
10369 0.3395    162
5009  0.3355    155
9917  0.3330    919
3867  0.3258    356
14276 0.2819    894

card1 transaction count distribution:
count   13553.0000
mean       43.5726
std       329.0845
min         1.0000
25%         1.0000
50%         4.0000
75%        14.0000
max     14932.0000
Name: count, dtype: float6

In [20]:
print("=== EMAIL DOMAIN ANALYSIS ===")

print("P_emaildomain fraud rates (top 15):")
email_stats = train.groupby("P_emaildomain")["isFraud"].agg(["mean", "count"])
email_stats = email_stats[email_stats["count"] >= 100].sort_values("mean", ascending=False)
print(email_stats.head(15))

print("\nEmail provider (extracted) fraud rates:")
train["P_email_provider"] = train["P_emaildomain"].str.split(".").str[0].fillna("unknown")
provider_stats = train.groupby("P_email_provider")["isFraud"].agg(["mean", "count"])
provider_stats = provider_stats[provider_stats["count"] >= 100].sort_values("mean", ascending=False)
print(provider_stats.head(10))


=== EMAIL DOMAIN ANALYSIS ===
P_emaildomain fraud rates (top 15):
                 mean   count
P_emaildomain                
mail.com       0.1896     559
outlook.es     0.1301     438
aim.com        0.1270     315
outlook.com    0.0946    5096
hotmail.es     0.0656     305
live.com.mx    0.0547     749
hotmail.com    0.0530   45250
gmail.com      0.0435  228355
yahoo.fr       0.0350     143
embarqmail.com 0.0346     260
mac.com        0.0321     436
icloud.com     0.0314    6267
comcast.net    0.0312    7888
charter.net    0.0306     816
frontier.com   0.0286     280

Email provider (extracted) fraud rates:
                   mean   count
P_email_provider               
mail             0.1896     559
aim              0.1270     315
outlook          0.0974    5534
hotmail          0.0525   46005
gmail            0.0435  228851
embarqmail       0.0346     260
live             0.0325    3846
mac              0.0321     436
icloud           0.0314    6267
comcast          0.0312    7888

In [22]:
print("=== PRODUCT ANALYSIS ===")
print("ProductCD fraud rates:")
print(train.groupby("ProductCD")["isFraud"].agg(["mean", "count"]).sort_values("mean", ascending=False))



=== PRODUCT ANALYSIS ===
ProductCD fraud rates:
            mean   count
ProductCD               
C         0.1169   68519
S         0.0590   11628
H         0.0477   33024
R         0.0378   37699
W         0.0204  439670


In [24]:
print("=== M COLUMNS (Match flags) ANALYSIS ===")
m_cols = [c for c in train.columns if c.startswith("M") and c[1:].isdigit()]
print(f"M columns: {m_cols}")

for col in m_cols:
    if col in train.columns:
        stats = train.groupby(col)["isFraud"].agg(["mean", "count"])
        print(f"\n{col} fraud rates:")
        print(stats)


=== M COLUMNS (Match flags) ANALYSIS ===
M columns: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']

M1 fraud rates:
     mean   count
M1               
F  0.0000      25
T  0.0199  319415

M2 fraud rates:
     mean   count
M2               
F  0.0349   33972
T  0.0181  285468

M3 fraud rates:
     mean   count
M3               
F  0.0303   67709
T  0.0171  251731

M4 fraud rates:
     mean   count
M4               
M0 0.0366  196405
M1 0.0271   52826
M2 0.1137   59865

M5 fraud rates:
     mean   count
M5               
F  0.0265  132491
T  0.0377  107567

M6 fraud rates:
     mean   count
M6               
F  0.0237  227856
T  0.0170  193324

M7 fraud rates:
     mean   count
M7               
F  0.0193  211374
T  0.0221   32901

M8 fraud rates:
     mean   count
M8               
F  0.0217  155251
T  0.0162   89037

M9 fraud rates:
     mean   count
M9               
F  0.0300   38632
T  0.0178  205656
